# 03. 전처리 — 정제 · 종목명 마스킹 · 형태소 분석 · 중복 제거 · 분할 · 골드셋 샘플링

핵심은 **종목명 마스킹**이다. 학습 기사는 전부 제목에 종목명이 있으므로, 가리지 않으면
모델이 "하이닉스 → 반도체"만 외우고 종목명 없는 기사에서는 아무것도 못 맞힌다.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from src.config import RAW, PROCESSED, GOLD, TAGS, MASK_TOKEN
from src.crawler import load_krx_listing
from src.dataset import build_weak_labels, dedupe, cap_per_tag, time_split, article_key, label_matrix
from src.masking import build_company_dict, CompanyMasker, unmasked_proper_nouns, default_masker
from src.config import EXTRA_MASK_NAMES
from src.text import clean, lead_sentences, tokenize_many
pd.set_option("display.max_colwidth", 120)

CAP_PER_TAG = 2500   # 태그별 상한: 흔한 태그가 학습을 지배하지 않도록
SEED = 42

d2_raw = pd.read_csv(RAW / "d2_stock_news_raw.csv", dtype={"code": str})
d3_raw = pd.read_csv(RAW / "d3_general_news_raw.csv")
listing = load_krx_listing()

## 1. 정제
HTML 태그·엔티티(API 단계에서 제거), 기자명·이메일, `[사진]`, `(서울=연합뉴스)` 머리말, 저작권 문구, URL 제거.
입력 텍스트는 **제목 + 요약 앞 4문장**(요약은 대부분 1~3문장이라 사실상 전체).

In [2]:
def prepare(df):
    df = df.copy()
    df["title_c"] = df.title.map(clean)
    df["desc_c"] = df.description.map(clean).map(lambda s: lead_sentences(s, 4))
    df["text_raw"] = df.title_c + " . " + df.desc_c
    df["pub_date"] = pd.to_datetime(df.pub_date)
    return df

ex = d2_raw.description.dropna()
ex = ex[ex.str.contains("기자|=|@|사진", regex=True)].head(3)
pd.DataFrame({"원문": ex.values, "정제 후": ex.map(clean).values})

,원문,정제 후
0,"[사진 = 유한양행 제공] ◇ 유한양행, 창립 100주년 기념 '유한양행 100년사' 발간 유한양행은 창립 100주년을 맞아 글로벌 제약사 도약을 위한 비전을 담은 '유한양행 100년사(유한 100년사)'를 ...","유한양행, 창립 100주년 기념 '유한양행 100년사' 발간 유한양행은 창립 100주년을 맞아 글로벌 제약사 도약을 위한 비전을 담은 '유한양행 100년사(유한 100년사)'를 발간했다고 20일 밝혔다. 유한..."
1,"유한양행 100년사. 사진=유한양행 유한양행은 창립 100주년을 맞아 '유한양행 100년사(이하 유한 100년사)'를 발간했다고 20일 밝혔다. 유한 100년사는 1926년 고(故) 유일한 박사가 ""건강한 ...","유한양행 100년사. 유한양행은 창립 100주년을 맞아 '유한양행 100년사(이하 유한 100년사)'를 발간했다고 20일 밝혔다. 유한 100년사는 1926년 고(故) 유일한 박사가 ""건강한 국민만이 주권을 ..."
2,"|중앙이코노미뉴스 최경서 기자|유한양행 100년사. [사진=유한양행] 유한양행이 창립 100주년을 맞아 기업의 성장 과정과 연구개발 성과, 사회적 책임의 역사를 담은 '유한양행 100년사'를 발간했다. 창업 ...","유한양행 100년사. 유한양행이 창립 100주년을 맞아 기업의 성장 과정과 연구개발 성과, 사회적 책임의 역사를 담은 '유한양행 100년사'를 발간했다. 창업 초기 국민필수..."


## 2. 약한 라벨 + 중복 제거
- 제목에 종목명/약칭이 있는 행만 → 같은 URL은 병합, 태그는 합집합 (02와 동일 규칙)
- 같은 URL, 그리고 정규화한 제목 앞 25자가 같은 기사(통신사 재전송·[속보] 머리말 차이) 제거

In [3]:
START = "2025-10-20"  # 수집 기간: 최근 12개월 (검색 API의 sim 정렬이 오래된 기사도 섞어 줌)
weak = prepare(build_weak_labels(d2_raw, title_filter=True))
print(f"기간 필터 전 {len(weak):,}건 (가장 오래된 기사 {weak.pub_date.min():%Y-%m-%d})")
weak = weak[weak.pub_date >= START]
n0 = len(weak)
weak = dedupe(weak)
print(f"약한 라벨 기사 {n0:,} → 중복 제거 후 {len(weak):,}")

# 비교 실험 E5용: 제목 필터 없이 만든 데이터
weak_nf = prepare(build_weak_labels(d2_raw, title_filter=False))
weak_nf = dedupe(weak_nf[weak_nf.pub_date >= START])
print(f"(E5) 제목 필터 없음: {len(weak_nf):,}")

기간 필터 전 161,656건 (가장 오래된 기사 2011-10-18)


약한 라벨 기사 125,625 → 중복 제거 후 114,565


(E5) 제목 필터 없음: 201,491


## 3. 종목명 마스킹
KRX 전체 상장사명(우선주 표기 제거) + 약칭·영문명 + 그룹명·해외 빅테크명을 사전으로 만들고 `[기업]`으로 바꾼다.
일반 명사와 겹치는 회사명(대상, 한국, 동방, 태양 …)은 사전에서 뺀다.

### 3-1. 1차 사전으로 마스킹 → 남은 고유명사 확인

In [4]:
names = build_company_dict(listing)
masker = CompanyMasker(names)
print("사전 크기:", len(names))
sample = weak.sample(3000, random_state=SEED)
masked_sample = sample.text_raw.map(masker.mask)
left = unmasked_proper_nouns(masked_sample, top=60)
print(", ".join(f"{w}({c})" for w, c in left.values))

사전 크기: 2935


AI(437), 미국(254), 서울(196), 한국(116), 뷰티(115), 유럽(99), 중국(96), 북미(91), 부산(74), LFP(72), 일본(72), ESG(70), 울산(65), MOU(62), 인천(62), 방산(51), KRX(49), LNG(49), 한국거래소(49), ESS(48), FDA(47), 베트남(43), 코스피(42), 인도(40), AX(38), ADC(35), 특징주(35), IR(32), 청주(32), 대전(32), 성수(31), MMORPG(31), 아시아(31), MBK(31), 멕시코(31), 이투데이(30), IP(30), 중동(30), HVDC(30), 경기(30), 경남(27), 경기도(27), 제주(27), 시공사(26), 파트너스(26), ETF(24), CEO(24), MW(23), SC(23), 이재근(23), 대한민국(22), IT(22), LCC(21), ODM(21), 중구(21), 송전(21), 평택(20), 모빌리티(20), 호주(20), CDMO(20)


### 3-2. 사전 보강 (사람이 고름)
위 표에서 **기업을 가리키는 고유명사**만 골라 추가한다. 지명·제품명·인명·기관명(정부 부처, 한국은행 등)은 남긴다.
제품·브랜드명(예: 갤럭시, 아이폰)은 산업 신호이므로 가리지 않는다.

In [5]:
# 고른 목록은 config.EXTRA_MASK_NAMES에 있다 (학습·적용·골드셋 평가에서 같은 마스커를 쓰도록)
print(EXTRA_MASK_NAMES)
masker = default_masker()
left2 = unmasked_proper_nouns(sample.text_raw.map(masker.mask), top=40)
print(", ".join(f"{w}({c})" for w, c in left2.values))

['삼전닉스', '하닉', '삼전', '현차', '엔솔', '네카토', '네카라쿠배', 'MBK', '쿠팡', '토스', '배민', '직방', 'SK온', '한국조선', '모비스', '에코프로머티', 'KB', '신한', 'LIG', 'DB', '핀다', '코카콜라', '웨스팅하우스', '카메코', '샌디스크', '웨스턴디지털', '창신메모리', 'CXMT', '화웨이', '샤오미', '퀄컴', 'AMD', 'ASML', '브로드컴', '폭스바겐', '도요타', '토요타', 'GM', '포드', 'BMW', '벤츠', '스텔란티스', '혼다', '닛산', '알리바바', '텐센트', '에어프레미아', '파라타항공', '듀프리']


AI(437), 미국(254), 서울(196), 한국(116), 뷰티(115), 유럽(99), 중국(96), 북미(91), 부산(74), LFP(72), 일본(72), ESG(70), 울산(65), MOU(62), 인천(62), 방산(51), KRX(49), LNG(49), 한국거래소(49), ESS(48), FDA(47), 베트남(43), 코스피(42), 인도(40), AX(38), ADC(35), 특징주(35), IR(32), 청주(32), 대전(32), 성수(31), MMORPG(31), 아시아(31), 멕시코(31), 이투데이(30), IP(30), 중동(30), HVDC(30), 경기(30), 경남(27)


In [6]:
weak["text"] = weak.text_raw.map(masker.mask)
weak["title_m"] = weak.title_c.map(masker.mask)
weak_nf["text"] = weak_nf.text_raw.map(masker.mask)
print("기사당 마스크 토큰 수 평균:", weak.text.str.count(r"\[기업\]").mean().round(2))
weak[["text_raw", "text"]].head(5)

기사당 마스크 토큰 수 평균: 3.44


,text_raw,text
0,"[단신]LG생건, 경주 APEC에 ‘휘오 울림워터’ 지원 . LG생건, 경주 APEC에 ‘휘오 울림워터’ 지원 LG생활건강이 이달 말 경북 경주에서 열리는 2025 아시아태평양경제협력체(APEC) 정상회의에 ...","[단신][기업], 경주 APEC에 ‘휘오 울림워터’ 지원 . [기업], 경주 APEC에 ‘휘오 울림워터’ 지원 [기업]이 이달 말 경북 경주에서 열리는 2025 아시아태평양경제협력체(APEC) 정상회의에 ‘휘..."
1,"[이번주 재개발ㆍ재건축] 한신공영, 창원 회원2구역 수주 유력 . 오는 25일 총회를 개최하며, 우선협상대상자로 지정된 한신공영을 시공사로 선정할 것으로 보인다. 이 사업은 마산회원구 회원동 480-31번지 ...","[이번주 재개발ㆍ재건축] [기업], 창원 회원2구역 수주 유력 . 오는 25일 총회를 개최하며, 우선협상대상자로 지정된 [기업]을 시공사로 선정할 것으로 보인다. 이 사업은 마산회원구 회원동 480-31번지 ..."
2,"[더벨][2025 이사회 평가]'LNG선' 사이클 탄 한국카본, 경영성과 '100점대... . 한국카본이 실적 개선과 주가 상승에 힘입어 theBoard '2025 이사회 평가' 점수가 큰폭으로 올라갔다. 지...","[더벨][2025 이사회 평가]'LNG선' 사이클 탄 [기업], 경영성과 '100점대... . [기업]이 실적 개선과 주가 상승에 힘입어 theBoard '2025 이사회 평가' 점수가 큰폭으로 올라갔다. 지..."
3,"풍산, 미국 관세만 구체화되면 매수 접근 유효-메리츠 . 메리츠증권이 20일 풍산에 대한 3분기 실적 전망치를 연결 기준 매출 1조1760억원, 영업이익 619억원으로... 관세는 풍산의 부산 안강공장 생산 ...","[기업], 미국 관세만 구체화되면 매수 접근 유효-메리츠 . [기업]이 20일 [기업]에 대한 3분기 실적 전망치를 연결 기준 매출 1조1760억원, 영업이익 619억원으로... 관세는 [기업]의 부산 안강공..."
4,"[더벨]SM엔터, '비핵심자산' 키이스트 340억에 매각 . 비핵심 자산을 정리하고 있는 SM엔터테인먼트(이하 SM엔터)가 마침내 손자회사인 키이스트를 매각한다. 수익성이 떨어지는 자산을 최소화하고 주력 사업...","[더벨][기업], '비핵심자산' [기업] 340억에 매각 . 비핵심 자산을 정리하고 있는 [기업](이하 [기업])가 마침내 손자회사인 [기업]를 매각한다. 수익성이 떨어지는 자산을 최소화하고 주력 사업인 음악..."


## 4. 태그별 상한 적용 (불균형 완화)
태그별로 최대 `CAP_PER_TAG`건. 희귀 태그 기사부터 확보한 뒤 흔한 태그를 채운다(다중 라벨 기사 때문에
흔한 태그는 상한을 약간 넘을 수 있다).

In [7]:
weak_c = cap_per_tag(weak, CAP_PER_TAG, seed=SEED)
weak_nf_c = cap_per_tag(weak_nf, CAP_PER_TAG, seed=SEED)
pd.DataFrame({"상한 전": pd.Series(label_matrix(weak.tag_list).sum(0), TAGS),
              "상한 후": pd.Series(label_matrix(weak_c.tag_list).sum(0), TAGS),
              "(E5) 필터 없음": pd.Series(label_matrix(weak_nf_c.tag_list).sum(0), TAGS)})

,상한 전,상한 후,(E5) 필터 없음
반도체,6666,2519,2917
2차전지,9174,2511,2535
자동차,8298,2500,2593
조선,3102,2504,2802
철강·비철,6712,2703,2801
원전·전력기기,8502,2503,2536
방산,4922,2657,3003
바이오·제약,10124,2502,2622
인터넷·게임,10000,2509,2537
금융(은행·증권·보험),13352,2500,2500


## 5. 형태소 분석 + 불용어 제거
`kiwipiepy`로 명사(NNG/NNP), 외국어(SL), 한자(SH), 어근(XR), 동사·형용사 어간(+'다')을 남긴다.
불용어: 일반 불용어 + 경제 뉴스 상투어(기자, 오늘, 전년, 대비, 밝히다 …) + 언론사명.
실험용으로 (a) 마스킹 O 본문, (b) 마스킹 X 본문(E2), (c) 제목만(E4) 세 가지를 만든다.

In [8]:
for df in (weak_c, weak_nf_c):
    df["tokens"] = tokenize_many(df.text.tolist())
weak_c["tokens_nomask"] = tokenize_many(weak_c.text_raw.tolist())
weak_c["tokens_title"] = tokenize_many(weak_c.title_m.tolist())
weak_c["title_raw_tokens"] = tokenize_many(weak_c.title_c.tolist())
weak_c[["text", "tokens"]].head(5)

,text,tokens
0,"[단신][기업], 경주 APEC에 ‘휘오 울림워터’ 지원 . [기업], 경주 APEC에 ‘휘오 울림워터’ 지원 [기업]이 이달 말 경북 경주에서 열리는 2025 아시아태평양경제협력체(APEC) 정상회의에 ‘휘...",단신 [기업] 경주 APEC 휘오 울림워터 지원 [기업] 경주 APEC 휘오 울림워터 지원 [기업] 이달 경북 경주 아시아태평양경제협력체 APEC 정상 회의 휘오 울림워터 지원 휘오 울림워터
1,"[기업], 미국 관세만 구체화되면 매수 접근 유효-메리츠 . [기업]이 20일 [기업]에 대한 3분기 실적 전망치를 연결 기준 매출 1조1760억원, 영업이익 619억원으로... 관세는 [기업]의 부산 안강공...",[기업] 미국 관세 구체 매수 접근 유효 메리츠 [기업] [기업] 분기 실적 전망 연결 기준 매출 영업 이익 관세 [기업] 부산 안강 공장 생산 구경 미국 자회사 PMC 판매 과정 발생
2,"[더벨][기업], '비핵심자산' [기업] 340억에 매각 . 비핵심 자산을 정리하고 있는 [기업](이하 [기업])가 마침내 손자회사인 [기업]를 매각한다. 수익성이 떨어지는 자산을 최소화하고 주력 사업인 음악...",더벨 [기업] 핵심 자산 [기업] 매각 핵심 자산 정리 [기업] [기업] 자회사 [기업] 매각 수익 떨어지다 자산 최소 주력 사업 음악 집중 선택 집중 전략 일환
3,"[기업], 영업이익 -45%...특수강 업황 침체 . [기업]의 3분기 영업이익은 비수기 영향 등으로 예상보다 부진할 전망이다. 최근 [기업]에 따르면 이 회사는 3분기 영업이익은 전년동기대비 45% 감소한 ...",[기업] 영업 이익 특수 업황 침체 [기업] 분기 영업 이익 비수기 영향 예상 부진 전망 [기업] 회사 분기 영업 이익 동기 감소 기존 추정 컨센서스
4,"[기업], 최대주주 변경…[기업] 합병 영향 . 글로벌 팬 플랫폼 운영사 [기업]의 최대주주가 [기업]로 변경된다. [기업]와... [기업] 측은 ""합병으로 인한 최대주주 변경이지만, 임원 선·해임 계획은 없...",[기업] 최대 주주 변경 [기업] 합병 영향 글로벌 플랫폼 운영 [기업] 최대 주주 [기업] 변경 [기업] [기업] 합병 인하다 최대 주주 변경 임원 해임 [기업] 연결 기준


In [9]:
for df in (weak_c, weak_nf_c):
    df["log_len"] = np.log1p(df.text.str.len())

## 6. 시간순 분할 (앞 80% 학습 / 뒤 20% 검증)

In [10]:
train, valid = time_split(weak_c, 0.8)
print(f"학습 {len(train):,}건 ({train.pub_date.min():%Y-%m-%d} ~ {train.pub_date.max():%Y-%m-%d})")
print(f"검증 {len(valid):,}건 ({valid.pub_date.min():%Y-%m-%d} ~ {valid.pub_date.max():%Y-%m-%d})")
pd.DataFrame({"train": label_matrix(train.tag_list).mean(0), "valid": label_matrix(valid.tag_list).mean(0)},
             index=TAGS).round(3).T

학습 28,738건 (2025-10-20 ~ 2026-10-07)
검증 7,185건 (2026-10-07 ~ 2026-10-20)


,반도체,2차전지,자동차,조선,철강·비철,원전·전력기기,방산,바이오·제약,인터넷·게임,금융(은행·증권·보험),건설·부동산,정유·화학,항공·여행,화장품·소비재,엔터·미디어
train,0.071,0.075,0.064,0.064,0.081,0.076,0.07,0.065,0.057,0.065,0.066,0.084,0.075,0.077,0.072
valid,0.066,0.048,0.092,0.092,0.054,0.046,0.09,0.089,0.121,0.088,0.085,0.032,0.050,0.039,0.063


검증셋에 태그 비율이 학습셋과 크게 다르면 (최근 이슈가 몰린 태그), 약한 라벨 검증 점수가 흔들릴 수 있다.
이 차이는 시간순 분할이 일부러 드러내는 '미래 분포 변화'다.

In [11]:
cols = ["key", "pub_date", "press", "title", "description", "title_c", "text_raw", "text", "title_m", "stocks",
        "tag_list", "n_tags", "tokens", "tokens_nomask", "tokens_title", "title_raw_tokens", "log_len"]
train[cols].to_pickle(PROCESSED / "weak_train.pkl")
valid[cols].to_pickle(PROCESSED / "weak_valid.pkl")
tr_nf, va_nf = time_split(weak_nf_c, 0.8)
tr_nf.to_pickle(PROCESSED / "weak_nofilter_train.pkl")
va_nf.to_pickle(PROCESSED / "weak_nofilter_valid.pkl")

## 7. D3: 종목명 없는 경제 뉴스
1. 정제, URL·제목 중복 제거
2. **종목명 사전(마스킹 사전과 동일)에 하나도 안 걸리는 기사만** 남김
3. 사전에 걸리는 비율이 높은 이름을 확인해, 일반 명사와 겹치는 회사명이 필터를 망가뜨리지 않는지 점검

In [12]:
d3 = prepare(d3_raw)
d3 = d3[d3.pub_date >= START]
d3["key"] = article_key(d3)
d3 = dedupe(d3)
# (D2 기사는 제목에 종목명이 있으므로 아래 '종목명 없음' 필터에서 자동으로 빠진다)
hits = d3.text_raw.map(masker.find)
print(f"D3 중복 제거 후 {len(d3):,}건, 기업명 포함 {hits.str.len().gt(0).mean():.1%}")
top_hits = pd.Series([h for hs in hits for h in set(hs)]).value_counts().head(40)
print(", ".join(f"{w}({c})" for w, c in top_hits.items()))

D3 중복 제거 후 104,889건, 기업명 포함 32.7%
삼성전자(1155), 삼성(1102), SK하이닉스(1002), 현대차(949), 대한항공(669), 포스코(668), 기아(624), 한화(514), 한화오션(461), HD현대(447), SK(416), 롯데(389), 삼성전자와(382), 엔비디아(354), 하나은행(351), TSMC(338), 삼전닉스(338), 에쓰오일(326), LS일렉트릭(321), 현대건설(318), HD현대중공업(305), 제주항공(300), 삼성물산(298), 신한(297), 신한은행(296), 쿠팡(290), 삼성전자는(286), 삼성중공업(276), 롯데면세점(272), 신세계(269), 애플(262), 웨스팅하우스(256), KAI(253), 마이크론(247), 현대제철(246), 삼성바이오로직스(242), GM(241), 효성중공업(239), LG(238), 구글(238)


위 목록은 대부분 실제 기업 지칭(그룹명·빅테크 등)이다. 일반 명사로 오인된 회사명이 있으면
`config.AMBIGUOUS_NAMES`에 추가하고 다시 실행한다 (1차 실행에서 확인 후 반영했다).

In [13]:
d3_clean = d3[hits.str.len() == 0].copy()
d3_clean["text"] = d3_clean.text_raw.map(masker.mask)  # 기업명은 없지만 경영자 이름 등은 학습과 같은 방식으로 가린다
d3_clean["tokens"] = tokenize_many(d3_clean.text.tolist())
d3_clean["log_len"] = np.log1p(d3_clean.text.str.len())
d3_clean = d3_clean[d3_clean.tokens.str.split().str.len() >= 5].reset_index(drop=True)
import hashlib
d3_clean["doc_id"] = "d3_" + d3_clean.key.map(lambda k: hashlib.md5(k.encode()).hexdigest()[:10])  # 사전을 고쳐도 ID 유지
print(f"종목명 없는 기사: {len(d3_clean):,}건")
print(d3_clean.groupby("query_group").size())
d3_clean[["doc_id", "pub_date", "query", "title_c", "desc_c", "text", "tokens", "press", "log_len", "query_group"]] \
    .to_pickle(PROCESSED / "d3_clean.pkl")
d3_clean.sample(5, random_state=SEED)[["query", "title_c", "desc_c"]]

종목명 없는 기사: 70,581건
query_group
industry    32934
macro       37647
dtype: int64


,query,title_c,desc_c
58698,소비 심리,"[현장] ""PF 300곳 밀착 케어·이주비 대출 확대""…정부, 주택공급 속도전...",전월세 시장 안정을 위한 주거안정망 구축…소비심리 진정세 시장 동향과 관련해 정부는 최근 서울 강남 3구를 중심으로 매매가 및 전세가 상승 폭이 줄어들고 있다고 분석했다. 서울 주택 매매시장 소비심리지수...
7267,보호무역,"""보호무역 장벽 넘는다""… KOTRA, 수출 기업 생존 위한 신통상 전략 제시","최근 세계 경제는 주요국 간의 패권 경쟁과 보호무역주의 확산으로 규제의 밀도가 촘촘해지는 양상이다. 미국의 산업정책 중심 공급망 재편, 유럽연합(EU)의 탄소국경조정제도(CBAM) 등 지속가능성 기반의 규범화..."
51814,OTT 시장,"JTBC, 한영회계법인 매각 주관사 확정…이달 공고","법원은 OTT 시장 성장과 방송광고 시장 축소, 콘텐츠 제작비 증가 등을 재무 악화의 주요 원인으로 꼽았다. 올림픽 중계권료 지출과 중앙그룹의 자금난도 영향을 미친 것으로 판단했다. 서울회생법원은 지난 8월 ..."
15538,엔터 업계,이름만 빼고 똑같다 … ETF 베끼기 논란,"금융당국이 유사 상품 출시와 관련해 업계에 자정 노력을 요구하고 있지만 실질적인 제재가 없는... 제약·바이오, 엔터 등에서 1개 산업을 추가해 5개 산업에 투자한다. 총 10개 종목에 압축 투자하며 반도체 ..."
46488,철강업계,[국정감사] EU 탄소국경조정제도 대비 中企 '공공플랫폼' 도입 시급,"특히 철강업계가 부담해야 할 EU CBAM 인증서 비용은 오는 2034년 한 해에만 5589억 원에 달할 것으로 추산되며, 2034년까지 누적 비용은 무려 2조 6000억 원에 이를 전망이다. 이로 인해 수출..."


## 8. 골드 평가셋 샘플링 (D4)
D3에서 무작위 300건 → 검증 100 + 테스트 200. 라벨링 기준: **"이 기사 내용 때문에 해당 산업 종목의 주가나
실적에 영향이 갈 수 있는가"** 를 기준으로 0~3개 태그, 해당 없으면 '없음'.
라벨링 파일: `data/gold/gold_labels.csv` = 표본 + `tags` 열(`|`로 구분, 빈칸 = 없음). 50건은 이중 라벨링용.
표본 파일에 기사 텍스트를 같이 저장해, 이후 사전을 보강해 D3 필터 결과가 바뀌어도 골드셋은 고정된다.

In [14]:
gold_path = GOLD / "gold_sample.csv"
if not gold_path.exists():  # 한 번 뽑은 표본은 고정 (라벨링 후 바뀌면 안 됨)
    g = d3_clean.sample(300, random_state=SEED).reset_index(drop=True)
    g["split"] = ["valid"] * 100 + ["test"] * 200
    g["double_label"] = [i < 50 for i in range(300)]
    g[["doc_id", "split", "double_label", "pub_date", "query", "press", "title_c", "desc_c", "text_raw"]].to_csv(
        gold_path, index=False, encoding="utf-8-sig")
gold_sample = pd.read_csv(gold_path)
gold_sample.split.value_counts()

split
test     200
valid    100
Name: count, dtype: int64

## 이 단계에서 알게 된 것

- **기간·중복**: 약한 라벨 기사 16.2만 건 중 최근 12개월(2025-10-20~) 기사 12.6만 건 → 통신사 재전송·[속보] 중복을
  제거하니 11.5만 건(약 9% 중복). 태그별 상한 2,500건을 적용해 학습 2.87만 / 약한 검증 7.2천 건으로 줄였다.
  검증셋은 시간순 마지막 20%라 2026-10-07 ~ 10-20, 약 2주다.
- **마스킹**: 기사당 평균 3.4개의 `[기업]` 토큰. 1차 사전만으로는 '하닉·삼전닉스' 같은 은어, 비상장 계열사
  (신한은행, 롯데면세점 등), 해외 기업명이 남았다. 그룹 접두어 + 계열사 접미사 규칙(`한화투자증권`, `DB인재개발원`)과
  추가 사전으로 보강한 뒤 남은 고유명사는 지명·기술 용어(AI, LFP, HVDC, FDA)·언론사명이 대부분이다.
- **D3 필터**: 경제 기사 10.5만 건(중복 제거 후) 중 32.7%가 어떤 기업명이든 언급한다. 이를 빼고 7.0만 건이 남았다
  (거시 검색어 3.8만, 산업 검색어 3.3만). 필터에 많이 걸린 이름은 삼성전자·SK하이닉스·현대차·그룹명·TSMC·엔비디아 순으로,
  일반 경제 기사도 대형 기업을 통해 산업을 이야기하는 경우가 많다는 뜻이다.
- **모호한 회사명**: '상보'(기사 표기), '배럴', '채비', '서산', 'DSR'(대출 규제), '이지스'(이지스함) 같은 이름이
  D3 필터를 망가뜨리고 있어 모호 목록에 넣었다. 사전 매칭은 정밀도와 재현율 사이 수작업 조정이 계속 필요하다.
- **골드셋**: D3에서 무작위 300건(검증 100 / 테스트 200)을 뽑아 라벨링했다. 표본 파일에 텍스트를 함께 저장해,
  이후 사전을 고쳐 D3 필터 결과가 바뀌어도 골드셋은 그대로 유지된다.